#  Memory in LLM Apps (Basic)

| **Concept** | **Purpose** |
|---|---|
| **Chat History** | Store current conversation |
| **Context Window** | Model's maximum available context |
| **Windowed Memory** | Keep recent messages |
| **Summarization** | Compress older messages |
| **Long-Term Memory** | Persist important information across conversations |

###  Easy Way to Remember Memory Concepts

| **Concept** | **Easy Meaning** |
|---|---|
| **Chat History** |  What happened? |
| **Context Window** |  How much can the model see? |
| **Windowed Memory** | Keep recent messages |
| **Summarization** |  Compress old messages |
| **Long-Term Memory** |  Remember important information |

LLM has **no memory by itself.** Every time, we send it the past messages again. All 5 concepts are just different ways of **deciding what to send.**

In [3]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)

### 1. Chat History

**Short note:** A list of all messages in the current conversation. We send the full list to the LLM every time, so it "remembers".

**Problem:** list keeps growing, cost goes up, and it can cross the limit.

In [13]:
history = []

def chat(user_msg):
    history.append({"role": "user", "content": user_msg})
    res = llm.invoke(history)
    history.append({"role": "ai", "content": res.content})
    return res.content

print(chat("Hello, my name is Mahesh and my wife name is Sejal"))
print(chat("What is my name?"))
print(chat("What is my wife name?"))
print(chat("Who is Sejal?"))
print(chat("Who is sejal husband?"))
print(len(history), "messages stored")

Hello Mahesh and Sejal! It's nice to meet you both. How can I assist you today?
Your name is Mahesh.
Your wife's name is Sejal.
Sejal is your wife's name.
Sejal is your wife's name, so your husband's name is Mahesh.
10 messages stored


### 2. Context Window

**Short note:** The maximum amount of text **(measured in tokens)** the model can read at once. Includes your history + new question + the answer.

--> Token ≈ small piece of a word (1 token ≈ 4 characters in English)

--> If history is bigger than the window → error, or old messages get cut

So we can't keep adding messages forever. That's why we need Windowed Memory and Summarization.

In [6]:
# check how many tokens our history is using
text = " ".join(m["content"] for m in history)
print("Tokens used:", llm.get_num_tokens(text))
print("Characters :", len(text))

c:\Users\Mahesh Yadav\Desktop\Gen-AI\venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\Mahesh Yadav\Desktop\Gen-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\Mahesh Yadav\Desktop\Gen-AI\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mahesh Yadav\.cache\

Tokens used: 39
Characters : 127


### 3. Windowed Memory

**Short note:** Send only the **last N messages.** Simple and cheap.

**Problem:** anything older than N is forgotten (like the user's name from message 1).

In [9]:
history = []
WINDOW = 4   # keep last 4 messages only

def chat_windowed(user_msg):
    history.append({"role": "user", "content": user_msg})

    recent = history[-WINDOW:]          # only the last 4
    res = llm.invoke(recent)

    history.append({"role": "ai", "content": res.content})
    return res.content

chat_windowed("Hi, my name is MAHESH")
chat_windowed("I like Python")
chat_windowed("I also like GEN-AI")
chat_windowed("Tell me one fun fact")
print(chat_windowed("What is my name?"))   # name is outside the window, so it may forget
print("FULL HISTORY:")
print(history)



I'm sorry, but as an AI language model, I don't have the capability to know your name. I'm just here to help you with any questions or tasks you might have.
FULL HISTORY:
[{'role': 'user', 'content': 'Hi, my name is MAHESH'}, {'role': 'ai', 'content': "Hello MAHESH! It's nice to meet you. How can I assist you today?"}, {'role': 'user', 'content': 'I like Python'}, {'role': 'ai', 'content': "That's great! Python is a versatile programming language that is widely used for web development, data analysis, artificial intelligence, and more. It's known for its readability and simplicity, which makes it a popular choice for beginners. If you're interested in learning more about Python, what would you like to know?"}, {'role': 'user', 'content': 'I also like GEN-AI'}, {'role': 'ai', 'content': "That's fantastic! AI (Artificial Intelligence) is an exciting field with a lot of potential applications. Python is indeed a great language for AI development due to its simplicity and the availability 

### 4. Summarization

**Short note:** Instead of throwing old messages away, **compress them into a short summary.** Send: summary + recent messages.

Keeps the important points, uses fewer tokens.

In [14]:
history = []
summary = ""
KEEP = 4   # keep last 4 messages as they are

def summarize(old_messages, old_summary):
    text = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)
    prompt = f"Existing summary: {old_summary}\n\nNew messages:\n{text}\n\n Write a short updated summary."
    return llm.invoke(prompt).content

def chat_summary(user_msg):
    global summary, history
    history.append({"role": "user", "content": user_msg})

    # if history is getting long, compress the old part
    if len(history) > KEEP + 2:
        old, history = history[:-KEEP], history[-KEEP:]
        summary = summarize(old, summary)

    messages = []
    if summary:
        messages.append({"role": "system", "content": f"Summary of earlier chat: {summary}"})
    messages += history

    res = llm.invoke(messages)
    history.append({"role": "ai", "content": res.content})
    return res.content

for m in ["Hi, my name is Mahesh", "I like Python", "I also like GEN-AI", "Tell me a fun fact", "Another one", "What is my name?"]:
    print("User:", m)
    print("AI  :", chat_summary(m), "\n")

print("SUMMARY:", summary)

User: Hi, my name is Mahesh
AI  : Hello Mahesh! It's nice to meet you. How can I assist you today? 

User: I like Python
AI  : That's great! Python is a versatile programming language that is widely used for web development, data analysis, artificial intelligence, and much more. It's known for its readability and simplicity, which makes it a popular choice for beginners. If you have any specific questions or need help with Python, feel free to ask! 

User: I also like GEN-AI
AI  : That's fantastic! GEN-AI, or General Artificial Intelligence, is a concept that aims to create a machine that can perform any intellectual task that a human can. While it's still a hypothetical goal, many researchers and companies are working on developing AI technologies that can approach or even surpass human capabilities in various domains. If you're interested in learning more about AI, including its potential applications and challenges, I'd be happy to help guide you through some resources. 

User: Tell

### 5. Long-Term Memory

**Short note:** Save important facts **outside the chat** (file / database), so a **new conversation** can still use them.

Flow: save fact → new chat → load facts → give to LLM

Below is the simplest version: a JSON file. (Real apps use a database or vector DB and pick only the relevant facts.)

In [16]:
import json, os

FILE = "memory.json"

def load_memory():
    return json.load(open(FILE)) if os.path.exists(FILE) else []

def save_memory(fact):
    facts = load_memory()
    if fact not in facts:
        facts.append(fact)
        json.dump(facts, open(FILE, "w"), indent=2)

# --- Conversation 1 (today) ---
save_memory("User's favorite language is python")
save_memory("User uses Google Gemini LLM")
# --- Conversation 2 (new chat, history is empty) ---
def chat_longterm(user_msg):
    facts = load_memory()
    system = "You know these facts about the user:\n" + "\n".join(f"- {f}" for f in facts)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    return llm.invoke(messages).content

print(chat_longterm("What programming language do I like?"))

Based on the information provided, it seems that you like both Java and Python. These are two popular programming languages that are widely used in the industry. Java is known for its platform independence and is often used for developing enterprise-level applications, while Python is popular for its simplicity and readability, making it great for scripting and rapid development. Both languages have their strengths and are used in various fields such as web development, mobile app development, and data science.


## Quick Summary

| **Concept** | **One Line** | **Code Idea** |
|---|---|---|
| **Chat History** | All messages of this chat | `history.append(...)` |
| **Context Window** | Model's limit | `llm.get_num_tokens(text)` |
| **Windowed Memory** | Last N messages | `history[-4:]` |
| **Summarization** | Old messages → short summary | `summary + recent` |
| **Long-Term Memory** | Facts saved in a file/DB | `save_memory()` / `load_memory()` |